In [0]:
'''
This script is about building the silver layer of Amazon product reviews. Goals will be:
- Clean and normalize text
- Standardize schema
- Remove obvious bad records
- Add NLP features
'''

'\nThis script is about building the silver layer of Amazon product reviews. Goals will be:\n- Clean and normalize text\n- Standardize schema\n- Remove obvious bad records\n- Add NLP features\n'

In [0]:
from pyspark.sql.functions import col, length, from_unixtime, lower, udf, regexp_replace, trim
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from pyspark.sql.types import DoubleType

In [0]:
bronze_df = spark.table("bronze_amazon_all_beauty")

In [0]:
# Step 2: Data Quality Rules
'''
Remove:
- missing review text
- missing rating
- duplicates
- extremely short reviews
'''
silver_df = bronze_df.filter(
    col("review_text").isNotNull() &
    col("rating").isNotNull()
)

silver_df = silver_df.filter(length("review_text") > 20)

#silver_df = silver_df.withColumn("review_id_str", col("review_id").cast("string"))
#silver_df = silver_df.dropDuplicates(["review_id_str"])

In [0]:
# Step 3: Normalize Timestamp
silver_df = silver_df.withColumn("review_ts", to_timestamp((col("review_timestamp") / 1000))
)

In [0]:
silver_df.show()

+--------------------+----------+--------------------+----------------+----------+--------------------+----------------+------+--------------------+
|             user_id|product_id|         review_text|review_timestamp|  category|        ingestion_ts|          source|rating|           review_ts|
+--------------------+----------+--------------------+----------------+----------+--------------------+----------------+------+--------------------+
|AGKHLEW2SOWHNMFQI...|B00YQ6X8EO|This spray is rea...|   1588687728923|All_Beauty|2026-03-01 18:05:...|All_Beauty.jsonl|   5.0|2020-05-05 14:08:...|
|AGKHLEW2SOWHNMFQI...|B081TJ8YS3|This product does...|   1588615855070|All_Beauty|2026-03-01 18:05:...|All_Beauty.jsonl|   4.0|2020-05-04 18:10:...|
|AE74DYR3QUGVPZJ3P...|B07PNNCSP9|Smells good, feel...|   1589665266052|All_Beauty|2026-03-01 18:05:...|All_Beauty.jsonl|   5.0|2020-05-16 21:41:...|
|AGMJ3EMDVL6OWBJF7...|B00R8DXL44|The polish was qu...|   1598567408138|All_Beauty|2026-03-01 18:05:...|All

In [0]:
# Step 4: Clean Text for NLP
silver_df = silver_df.withColumn(
    "clean_text",
    trim(
        regexp_replace(
            lower(col("review_text")),
            "[^a-zA-Z0-9\\s]",
            ""
        )
    )
)

In [0]:
# Step 5: Feature Engineering
silver_df = silver_df.withColumn(
    "review_length",
    length("clean_text")
)

In [0]:
# Step 6: Add Sentiment Score

analyzer = SentimentIntensityAnalyzer()

def get_sentiment(text):
    if text:
        return float(analyzer.polarity_scores(text)["compound"])
    return 0.0

sentiment_udf = udf(get_sentiment, DoubleType())

silver_df = silver_df.withColumn(
    "sentiment_score",
    sentiment_udf(col("clean_text"))
)

In [0]:
# Step 7: Select Final Schema

silver_df = silver_df.select(
    "product_id",
    "category",
    "review_ts",
    "rating",
    "review_text",
    "clean_text",
    "review_length",
    "sentiment_score",
    "ingestion_ts",
    "source"
)

In [0]:
silver_df.show()

+----------+----------+--------------------+------+--------------------+--------------------+-------------+---------------+--------------------+----------------+
|product_id|  category|           review_ts|rating|         review_text|          clean_text|review_length|sentiment_score|        ingestion_ts|          source|
+----------+----------+--------------------+------+--------------------+--------------------+-------------+---------------+--------------------+----------------+
|B00YQ6X8EO|All_Beauty|2020-05-05 14:08:...|   5.0|This spray is rea...|this spray is rea...|          290|         0.8008|2026-03-01 18:05:...|All_Beauty.jsonl|
|B081TJ8YS3|All_Beauty|2020-05-04 18:10:...|   4.0|This product does...|this product does...|          229|         0.7783|2026-03-01 18:05:...|All_Beauty.jsonl|
|B07PNNCSP9|All_Beauty|2020-05-16 21:41:...|   5.0|Smells good, feel...|smells good feels...|           23|         0.7906|2026-03-01 18:05:...|All_Beauty.jsonl|
|B00R8DXL44|All_Beauty|2020-

In [0]:
# Step 8: Write Silver Table
silver_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_amazon_reviews")

In [0]:
spark.sql("SELECT COUNT(*) FROM silver_amazon_reviews").show()

+--------+
|COUNT(*)|
+--------+
|  622643|
+--------+



In [0]:
# OPTIMIZE silver_amazon_reviews;
# ZORDER BY (product_id);